<a href="https://colab.research.google.com/github/charannideeshvm/LifeilotAI/blob/main/nids_v12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RansomGuard: Deep Learning Attack Detection

**How it works (in 4 lines)**
1. An **autoencoder** is trained only on normal traffic. It learns to copy normal flows. Strange flows are copied badly, so they get a high **error**.
2. A **classifier** (a small neural network) is trained on labeled flows to say which type each flow is (normal, DDoS, PortScan, ...).
3. The **fused classifier** also receives the autoencoder's error as one extra input.
4. We test on *later* traffic than we trained on.

## 1. Imports

In [1]:
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cpu


**Report helper file.** Small functions that build the report tables. The notebook and the app both use this one file.

In [2]:
%%writefile report_utils.py
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score


def percent(part, whole):
    """part / whole as text, for example '96.60%' ('n/a' when whole is 0)."""
    return "n/a" if whole == 0 else f"{part / whole:.2%}"


def category_tables(true, pred, labels):
    """Returns (table, matrix).
    table  = one row per category: TP, FN, FP, TN, precision, recall, F1.
    matrix = confusion matrix: rows = ground truth, columns = predicted."""
    matrix = confusion_matrix(true, pred, labels=labels)
    total = matrix.sum()
    rows = []
    for i, name in enumerate(labels):
        tp = matrix[i, i]                       # this category, named correctly
        fn = matrix[i, :].sum() - tp            # this category, named as something else
        fp = matrix[:, i].sum() - tp            # other category, named as this one
        tn = total - tp - fn - fp               # everything else
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        rows.append([name, tp + fn, tp, fn, fp, tn, round(precision, 4), round(recall, 4), round(f1, 4)])
    table = pd.DataFrame(rows, columns=["Category", "Flows (ground truth)", "TP", "FN", "FP", "TN", "Precision", "Recall", "F1"])
    matrix = pd.DataFrame(matrix, index=labels, columns=labels)
    matrix.index.name = "Ground truth (rows) / Predicted (columns)"
    return table, matrix


def overall_table(true, pred):
    """Headline numbers: accuracy, attacks caught, false alarms, precision, macro F1."""
    true, pred = np.asarray(true), np.asarray(pred)
    real_attack = true != "BENIGN"
    flagged = pred != "BENIGN"
    rows = [
        ["Flows with ground truth", f"{len(true):,}"],
        ["Accuracy (category named exactly right)", percent((true == pred).sum(), len(true))],
        ["Attacks caught (real attacks that were flagged)", percent((flagged & real_attack).sum(), real_attack.sum())],
        ["False alarms (normal flows that were flagged)", percent((flagged & ~real_attack).sum(), (~real_attack).sum())],
        ["Alerts that were real attacks (precision)", percent((flagged & real_attack).sum(), flagged.sum())],
        ["Macro F1 (average over categories in the ground truth)",
         f"{f1_score(true, pred, labels=np.unique(true), average='macro', zero_division=0):.3f}"],
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


def definitions_table():
    """What each metric in the report means."""
    rows = [
        ["Accuracy", "Share of flows whose category was named exactly right."],
        ["Attacks caught", "Real attack flows that were flagged (predicted as anything except BENIGN), divided by all real attack flows. Also called attack recall."],
        ["False alarms", "Normal (BENIGN) flows that were flagged, divided by all normal flows."],
        ["Alerts that were real attacks", "Flagged flows that really are attacks, divided by all flagged flows (precision of the attack group)."],
        ["Macro F1", "Average of the F1 scores of the categories, so every category counts equally."],
        ["TP (true positive)", "Flows of this category that the model named correctly."],
        ["FN (false negative)", "Flows of this category that the model named as something else."],
        ["FP (false positive)", "Flows of other categories that the model named as this category."],
        ["TN (true negative)", "Flows of other categories that the model did not name as this category."],
        ["Precision", "TP / (TP + FP): when the model says this category, how often it is right."],
        ["Recall", "TP / (TP + FN): how many flows of this category the model finds."],
        ["F1", "Balance of precision and recall: 2 x precision x recall / (precision + recall)."],
        ["Confusion matrix", "Rows are the ground truth, columns are the model's prediction. The diagonal holds correct answers; every other cell is a mix-up."],
    ]
    return pd.DataFrame(rows, columns=["Metric", "What it means"])

Writing report_utils.py


In [3]:
from report_utils import category_tables, overall_table

## 2. Load the data
For each type of traffic in each file: the first 70% of its rows are for **training**, the last 30% are for **testing**.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
folder = "/content/drive/MyDrive/dataset_nids/"
train_parts = []
test_parts = []

for path in sorted(glob.glob(folder + "*.csv")):
    df = pd.read_csv(path, encoding_errors="replace")

    # Clean: tidy column names, drop broken rows, tidy label text
    df.columns = df.columns.str.strip()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df["Label"] = df["Label"].str.strip().str.replace(r"[^\x00-\x7f]+", "-", regex=True)

    # Split each traffic type (BENIGN, DDoS, ...) into earlier rows and later rows
    for label, group in df.groupby("Label", sort=False):
        cut = int(len(group) * 0.7)
        train_parts.append(group.iloc[:cut])
        test_parts.append(group.iloc[cut:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)
print("Train flows:", len(train_df), "| Test flows:", len(test_df))

Train flows: 1979503 | Test flows: 848373


## 3. Prepare the data

In [6]:
# Feature columns = everything except the label
feature_names = [c for c in train_df.columns if c != "Label"]

# List of traffic types. BENIGN (normal) is always number 0.
all_labels = set(train_df["Label"]) | set(test_df["Label"])
classes = ["BENIGN"] + sorted(all_labels - {"BENIGN"})
print(classes)

# Numbers (X) and names (names) for train and test
X_train = train_df[feature_names].values.astype("float32")
names_train = train_df["Label"].values
X_test = test_df[feature_names].values.astype("float32")
names_test = test_df["Label"].values

# Training data: normal flows and attack flows
is_normal = names_train == "BENIGN"
X_attack = X_train[~is_normal]
names_attack = names_train[~is_normal]

# Normal flows are split in two halves: one half trains the autoencoder,
# the other half is kept for the classifier
X_normal_ae, X_normal_clf = train_test_split(X_train[is_normal], train_size=0.5, random_state=42)

['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


**Scaling:** put all features on a similar scale. The scaler learns from the autoencoder's normal data only. `clip` limits extreme values.

In [7]:
scaler = StandardScaler().fit(X_normal_ae)

def scale(X):
    X = scaler.transform(X)
    X = np.clip(X, -10, 10)
    return X.astype("float32")

X_normal_ae = scale(X_normal_ae)
X_normal_clf = scale(X_normal_clf)
X_attack = scale(X_attack)
X_test = scale(X_test)

# Free memory (the big tables are no longer needed)
del train_df, test_df, train_parts, test_parts, X_train, df, group

## 4. Autoencoder
It squeezes the 78 features down to 32 numbers, then tries to rebuild the 78. It practices on normal traffic only.

In [8]:
class Autoencoder(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, n_features))

    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder(len(feature_names)).to(device)

def get_errors(X):
    """For each flow: how badly was it rebuilt? High = strange."""
    X = torch.tensor(X).to(device)
    with torch.no_grad():
        rebuilt = ae(X)
    return ((X - rebuilt) ** 2).mean(dim=1).cpu().numpy()

**Training.** One *epoch* = one pass over all the normal flows, 512 flows at a time. After each epoch we measure the error on normal flows the autoencoder has not practiced on, and we keep the best version.

In [9]:
optimizer = torch.optim.Adam(ae.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
train_data = torch.tensor(X_normal_ae)

best_error = 999

for epoch in range(30):
    order = torch.randperm(len(train_data))              # shuffle
    for i in range(0, len(order), 512):
        batch = train_data[order[i:i + 512]].to(device)
        loss = loss_fn(ae(batch), batch)                 # how bad is the rebuild?
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()                                 # improve a little

    error = get_errors(X_normal_clf).mean()
    print(f"Epoch {epoch + 1} | error on unseen normal flows: {error:.5f}")

    if error < best_error:                               # keep the best version
        best_error = error
        torch.save(ae.state_dict(), "best_ae.pth")

ae.load_state_dict(torch.load("best_ae.pth"))

# Threshold for "autoencoder alone": only 1% of normal flows go above it
threshold = np.percentile(get_errors(X_normal_clf), 99)

Epoch 1 | error on unseen normal flows: 0.00671
Epoch 2 | error on unseen normal flows: 0.00386
Epoch 3 | error on unseen normal flows: 0.00261
Epoch 4 | error on unseen normal flows: 0.00199
Epoch 5 | error on unseen normal flows: 0.00167
Epoch 6 | error on unseen normal flows: 0.00143
Epoch 7 | error on unseen normal flows: 0.00130
Epoch 8 | error on unseen normal flows: 0.00113
Epoch 9 | error on unseen normal flows: 0.00111
Epoch 10 | error on unseen normal flows: 0.00097
Epoch 11 | error on unseen normal flows: 0.00092
Epoch 12 | error on unseen normal flows: 0.00092
Epoch 13 | error on unseen normal flows: 0.00092
Epoch 14 | error on unseen normal flows: 0.00077
Epoch 15 | error on unseen normal flows: 0.00094
Epoch 16 | error on unseen normal flows: 0.00068
Epoch 17 | error on unseen normal flows: 0.00069
Epoch 18 | error on unseen normal flows: 0.00079
Epoch 19 | error on unseen normal flows: 0.00069
Epoch 20 | error on unseen normal flows: 0.00061
Epoch 21 | error on unseen no

## 5. Classifiers
A small neural network. For each flow it gives one score per traffic type, and the biggest score is the answer.

Rare attack types get a slightly bigger weight, so the network does not ignore them.

In [10]:
def build_classifier(n_inputs, n_classes):
    return nn.Sequential(nn.Linear(n_inputs, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes))

def train_classifier(X, y, epochs=8):
    model = build_classifier(X.shape[1], len(classes)).to(device)

    # Weight per traffic type: rarer type = bigger weight
    counts = np.bincount(y, minlength=len(classes)) + 1
    weights = torch.tensor(1 / counts ** 0.25, dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    X = torch.tensor(X).to(device)
    y = torch.tensor(y).to(device)

    for epoch in range(epochs):
        order = torch.randperm(len(X), device=device)    # shuffle
        for i in range(0, len(order), 1024):
            pick = order[i:i + 1024]
            loss = loss_fn(model(X[pick]), y[pick])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model

def predict(model, X):
    """Returns the traffic type number for each flow (0 = BENIGN)."""
    with torch.no_grad():
        scores = model(torch.tensor(X).to(device))
    return scores.argmax(dim=1).cpu().numpy()

**The autoencoder clue** = how strange the autoencoder finds a flow (its error, on a log scale, standardized). The fused classifier gets it as 1 extra input: 79 numbers instead of 78.

In [11]:
# Classifier training data: unseen normal flows + attack flows
X_clf = np.vstack([X_normal_clf, X_attack])
names_clf = np.concatenate([["BENIGN"] * len(X_normal_clf), names_attack])
y_clf = np.array([classes.index(name) for name in names_clf])

# Standardize the clue using the classifier's training flows
clue_train = np.log10(get_errors(X_clf) + 1e-6)
clue_mean, clue_std = clue_train.mean(), clue_train.std()

def add_clue(X):
    """Add the autoencoder clue as one extra column."""
    clue = np.log10(get_errors(X) + 1e-6)
    clue = (clue - clue_mean) / clue_std
    return np.hstack([X, clue[:, None]]).astype("float32")

F_clf = add_clue(X_clf)
F_test = add_clue(X_test)

In [12]:
mlp = train_classifier(X_clf, y_clf)         # 78 features only
fused_mlp = train_classifier(F_clf, y_clf)   # 78 features + autoencoder clue
print("Trained")

Trained


## 6. Results on the later 30% of traffic
*Attacks caught* = share of attacks that were flagged. *False alarms* = share of normal flows wrongly flagged.

In [13]:
is_attack_test = names_test != "BENIGN"

# True = the model flags this flow as an attack
ae_flag = get_errors(X_test) > threshold
mlp_flag = predict(mlp, X_test) != 0
fused_flag = predict(fused_mlp, F_test) != 0

def caught(flag):
    return flag[is_attack_test].mean()

def false_alarms(flag):
    return flag[~is_attack_test].mean()

print(f"AE alone   | attacks caught: {caught(ae_flag):.1%} | false alarms: {false_alarms(ae_flag):.2%}")
print(f"MLP        | attacks caught: {caught(mlp_flag):.1%} | false alarms: {false_alarms(mlp_flag):.2%}")
print(f"Fused MLP  | attacks caught: {caught(fused_flag):.1%} | false alarms: {false_alarms(fused_flag):.2%}")

AE alone   | attacks caught: 36.3% | false alarms: 0.56%
MLP        | attacks caught: 96.3% | false alarms: 1.60%
Fused MLP  | attacks caught: 96.4% | false alarms: 1.48%


Share of each attack type that was flagged:

In [14]:
print(f"{'Attack type':28s} {'AE':>7s} {'MLP':>7s} {'Fused':>7s}")
for attack in classes[1:]:
    rows = names_test == attack
    print(f"{attack:28s} {ae_flag[rows].mean():7.1%} {mlp_flag[rows].mean():7.1%} {fused_flag[rows].mean():7.1%}   (n={rows.sum()})")

Attack type                       AE     MLP   Fused
Bot                             0.0%    8.0%    9.5%   (n=587)
DDoS                           55.1%   98.8%   99.0%   (n=38408)
DoS GoldenEye                  47.8%   97.2%   96.3%   (n=3088)
DoS Hulk                       52.9%   93.7%   93.8%   (n=69038)
DoS Slowhttptest               32.5%   78.1%   78.0%   (n=1650)
DoS slowloris                  40.3%   86.9%   96.4%   (n=1739)
FTP-Patator                     0.1%   97.0%   99.6%   (n=2381)
Heartbleed                    100.0%   75.0%  100.0%   (n=4)
Infiltration                   27.3%   54.5%   45.5%   (n=11)
PortScan                        0.3%   99.8%   99.4%   (n=47642)
SSH-Patator                     0.0%   98.6%   99.0%   (n=1770)
Web Attack - Brute Force        0.0%   99.6%   99.6%   (n=453)
Web Attack - Sql Injection      0.0%   42.9%   42.9%   (n=7)
Web Attack - XSS                2.0%   97.4%   94.4%   (n=196)


## 7. Extra checks

**Per-type report** for the fused model: which types does it get right, and which does it mix up?

In [15]:
from sklearn.metrics import classification_report

predicted_names = np.array(classes)[predict(fused_mlp, F_test)]
print(classification_report(names_test, predicted_names, zero_division=0))

                            precision    recall  f1-score   support

                    BENIGN       0.99      0.99      0.99    681399
                       Bot       0.05      0.10      0.07       587
                      DDoS       1.00      0.99      0.99     38408
             DoS GoldenEye       0.92      0.96      0.94      3088
                  DoS Hulk       0.98      0.94      0.96     69038
          DoS Slowhttptest       0.61      0.77      0.68      1650
             DoS slowloris       0.92      0.70      0.80      1739
               FTP-Patator       0.97      0.99      0.98      2381
                Heartbleed       1.00      0.75      0.86         4
              Infiltration       0.67      0.36      0.47        11
                  PortScan       0.88      0.99      0.94     47642
               SSH-Patator       0.82      0.99      0.90      1770
  Web Attack - Brute Force       0.30      0.90      0.45       453
Web Attack - Sql Injection       0.00      0.00

**Realistic traffic:** real networks are mostly normal. If attacks were only 10% or 1% of flows, how many alerts would be real attacks?

In [16]:
for share in [0.10, 0.01]:
    print(f"\nIf attacks are {share:.0%} of flows:")
    for name, flag in [("AE alone", ae_flag), ("MLP", mlp_flag), ("Fused MLP", fused_flag)]:
        real = caught(flag) * share                      # real attacks flagged
        fake = false_alarms(flag) * (1 - share)          # normal flows flagged
        print(f"  {name:10s} {real / (real + fake):.1%} of alerts are real attacks")


If attacks are 10% of flows:
  AE alone   87.8% of alerts are real attacks
  MLP        87.0% of alerts are real attacks
  Fused MLP  87.9% of alerts are real attacks

If attacks are 1% of flows:
  AE alone   39.6% of alerts are real attacks
  MLP        37.8% of alerts are real attacks
  Fused MLP  39.7% of alerts are real attacks


**Zero-day test:** remove one attack type from training, then see who still catches it. This is what the autoencoder is for.

In [17]:
print(f"{'Unseen attack':15s} {'AE':>7s} {'MLP':>7s} {'Fused':>7s}")
for unseen in ["PortScan", "DoS Hulk", "Bot"]:
    X_unseen = X_test[names_test == unseen][:20000]

    # Train new classifiers WITHOUT this attack type
    keep = names_clf != unseen
    mlp_zd = train_classifier(X_clf[keep], y_clf[keep])
    fused_zd = train_classifier(F_clf[keep], y_clf[keep])

    ae_rate = (get_errors(X_unseen) > threshold).mean()
    mlp_rate = (predict(mlp_zd, X_unseen) != 0).mean()
    fused_rate = (predict(fused_zd, add_clue(X_unseen)) != 0).mean()
    print(f"{unseen:15s} {ae_rate:7.1%} {mlp_rate:7.1%} {fused_rate:7.1%}")

Unseen attack        AE     MLP   Fused
PortScan           0.2%    0.0%    0.0%
DoS Hulk          71.3%   46.8%   45.4%
Bot                0.0%    0.0%    0.0%


## 8. Demo: 8 normal flows and 7 attack flows

In [18]:
normal_ids = np.where(~is_attack_test)[0][:8]
attack_ids = np.random.choice(np.where(is_attack_test)[0], 7, replace=False)
ids = np.concatenate([normal_ids, attack_ids])

predicted = predict(fused_mlp, F_test[ids])

for i in range(len(ids)):
    action = "allow" if predicted[i] == 0 else "BLOCK (simulated)"
    print(f"true: {names_test[ids[i]]:26s} predicted: {classes[predicted[i]]:26s} -> {action}")

true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: BENIGN                     predicted: BENIGN                     -> allow
true: PortScan                   predicted: PortScan                   -> BLOCK (simulated)
true: PortScan                   predicted: PortScan                   -> BLOCK (simulated)
true: DoS Hulk                   predicted: BENIGN                     -> allow
true: DDoS                       predicted: DDoS                       -> BLOCK (simulated)
true

## 9. Save everything the app needs
This includes the held-out test results, so the app's report can show them next to the results for an uploaded file.

In [19]:
# Results on the held-out test split, saved so the app's report can include them
test_categories, test_matrix = category_tables(names_test, predicted_names, classes)
test_overall = overall_table(names_test, predicted_names)

split_info = pd.DataFrame({
    "Item": ["Dataset", "Split rule", "Training flows (total)", "   used to train the autoencoder (normal only)",
             "   used to train the classifier (normal + attacks)", "Test flows (never trained on)",
             "Features per flow", "Categories", "Model"],
    "Value": ["CIC-IDS2017 (8 CSV files)",
              "For each traffic type in each file: first 70% of rows = training, last 30% = testing",
              f"{len(X_normal_ae) + len(X_clf):,}", f"{len(X_normal_ae):,}", f"{len(X_clf):,}", f"{len(X_test):,}",
              len(feature_names), len(classes), "Fused classifier: 78 features + autoencoder error"],
})

# Train / test split per category (for the report)
split_table = pd.DataFrame({"Category": classes})
split_table["Training flows"] = [int((names_train == c).sum()) for c in classes]
split_table["Test flows"] = [int((names_test == c).sum()) for c in classes]
split_table["Total"] = split_table["Training flows"] + split_table["Test flows"]
split_table["Test share"] = (split_table["Test flows"] / split_table["Total"]).map("{:.1%}".format)
total = split_table[["Training flows", "Test flows", "Total"]].sum()
split_table.loc[len(split_table)] = ["ALL CATEGORIES", *total, f"{total['Test flows'] / total['Total']:.1%}"]

torch.save(ae.state_dict(), "ransomguard_autoencoder.pth")
torch.save(fused_mlp.state_dict(), "ransomguard_classifier.pth")
joblib.dump({"scaler": scaler, "columns": feature_names, "classes": classes,
             "clue_mean": clue_mean, "clue_std": clue_std,
             "test_overall": test_overall, "test_categories": test_categories,
             "test_matrix": test_matrix, "split_info": split_info, "split_table": split_table}, "ransomguard_meta.pkl")
print("Saved")

Saved


## 10. Web app (Streamlit)
The app repeats the same steps: scale, autoencoder clue, classifier. Its **Download full report** button gives one Excel file with 10 sheets: summary, metrics, definitions, train/test split, results per category and confusion matrices (for the upload and for the held-out test), and the predicted output.

In [22]:
%%writefile app.py
import io
import streamlit as st
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from report_utils import category_tables, overall_table, definitions_table

class Autoencoder(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, n_features))

    def forward(self, x):
        return self.decoder(self.encoder(x))

def build_classifier(n_inputs, n_classes):
    return nn.Sequential(nn.Linear(n_inputs, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes))

# Load what the notebook saved
meta = joblib.load("ransomguard_meta.pkl")
scaler, columns, classes = meta["scaler"], meta["columns"], meta["classes"]

ae = Autoencoder(len(columns))
ae.load_state_dict(torch.load("ransomguard_autoencoder.pth", map_location="cpu"))
ae.eval()

clf = build_classifier(len(columns) + 1, len(classes))
clf.load_state_dict(torch.load("ransomguard_classifier.pth", map_location="cpu"))
clf.eval()

def predict_classes(X):
    """Same steps as the notebook: scale -> autoencoder clue -> classifier."""
    X = np.clip(scaler.transform(X), -10, 10).astype("float32")
    with torch.no_grad():
        batch = torch.tensor(X)
        error = ((batch - ae(batch)) ** 2).mean(dim=1).numpy()
        clue = (np.log10(error + 1e-6) - meta["clue_mean"]) / meta["clue_std"]
        fused = torch.tensor(np.hstack([X, clue[:, None]]).astype("float32"))
        return clf(fused).argmax(dim=1).numpy()

def make_report(file_name, predicted, truth):
    """Build the Excel report (10 sheets) and return it as bytes.
    predicted = category name for every flow; truth = ground-truth names, or None."""
    labels = list(classes)
    if truth is not None:
        labels += sorted(set(truth) - set(classes))      # labels the model was never trained on
    no_truth = pd.DataFrame({"Note": ["No ground truth: the uploaded file has no Label column."]})
    is_attack = predicted != "BENIGN"

    # 1. Summary of this scan
    summary = pd.DataFrame({
        "Item": ["File", "Total flows", "Flagged as attack", "Allowed", "Attack %", "Ground truth supplied"],
        "Value": [file_name, f"{len(predicted):,}", f"{is_attack.sum():,}", f"{(~is_attack).sum():,}",
                  f"{is_attack.mean():.2%}", "Yes" if truth is not None else "No"],
    })

    # 2. Metrics for this upload
    if truth is not None:
        metrics = overall_table(truth, predicted).rename(columns={"Value": "This upload"})
    else:
        metrics = pd.DataFrame({"Metric": ["No Label column in file"], "This upload": ["n/a"]})

    # 4. Train / test split and model
    split = meta["split_info"].copy()
    split.loc[len(split)] = ["Uploaded file", "Not split: every flow in the upload is predicted. CIC-IDS2017 files are about 70% training data, so use sheets 7 and 8 for unbiased accuracy."]

    # 5 and 6. This upload, per category
    if truth is not None:
        upload_categories, upload_matrix = category_tables(truth, predicted, labels)

    # 9. Predicted output: flows per category
    counts = pd.DataFrame({"Category": labels, "Flows predicted": [int((predicted == c).sum()) for c in labels]})
    if truth is not None:
        counts["Flows in ground truth"] = [int((truth == c).sum()) for c in labels]

    # 10. Predicted output: 1,000 random flows
    ids = np.sort(np.random.default_rng(0).choice(len(predicted), min(1000, len(predicted)), replace=False))
    sample = pd.DataFrame({"Row in uploaded file": ids + 1, "Predicted category": predicted[ids]})
    if truth is not None:
        sample.insert(1, "Ground truth", truth[ids])
        sample["Correct?"] = np.where(truth[ids] == predicted[ids], "Yes", "No")
    sample["Action"] = np.where(predicted[ids] == "BENIGN", "allow", "BLOCK (simulated)")

    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="1 Summary", index=False)
        metrics.to_excel(writer, sheet_name="2 Metrics", index=False)
        definitions_table().to_excel(writer, sheet_name="3 Definitions", index=False)
        split.to_excel(writer, sheet_name="4 Split and model", index=False)
        meta["split_table"].to_excel(writer, sheet_name="4 Split and model", index=False, startcol=3)
        if truth is not None:
            upload_categories.to_excel(writer, sheet_name="5 Upload per category", index=False)
            upload_matrix.to_excel(writer, sheet_name="6 Upload confusion matrix")
        else:
            no_truth.to_excel(writer, sheet_name="5 Upload per category", index=False)
            no_truth.to_excel(writer, sheet_name="6 Upload confusion matrix", index=False)
        meta["test_categories"].to_excel(writer, sheet_name="7 Test per category", index=False)
        meta["test_matrix"].to_excel(writer, sheet_name="8 Test confusion matrix")
        counts.to_excel(writer, sheet_name="9 Predicted counts", index=False)
        sample.to_excel(writer, sheet_name="10 Predicted sample", index=False)

        # Make every column wide enough to read
        for sheet in writer.sheets.values():
            for column in sheet.columns:
                longest = max((len(str(cell.value)) for cell in column if cell.value is not None), default=0)
                sheet.column_dimensions[column[0].column_letter].width = min(longest + 2, 90)
    return buffer.getvalue()

# ---- Screen ----
st.title("🛡️ RansomGuard: Network Attack Detection")
file = st.file_uploader("Upload a CIC-IDS2017 file", type=["csv", "parquet"])

if file is not None:
    if file.name.endswith(".parquet"):
        data = pd.read_parquet(file)
    else:
        data = pd.read_csv(file)
    data.columns = data.columns.str.strip()
    data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    # Some parquet files drop this duplicate column; it is an exact copy
    if "Fwd Header Length.1" in columns and "Fwd Header Length.1" not in data.columns:
        data["Fwd Header Length.1"] = data["Fwd Header Length"]

    # Any other missing column is filled with the training average
    missing = [c for c in columns if c not in data.columns]
    if missing:
        st.warning(f"Missing columns filled with training average (less accurate): {missing}")
        for c in missing:
            data[c] = scaler.mean_[columns.index(c)]

    X = data[columns].values.astype("float64")
    st.write("Flows loaded:", len(X))

    if st.button("Run scan"):
        predicted = np.array(classes)[predict_classes(X)]
        is_attack = predicted != "BENIGN"

        # Ground truth = the Label column, if the file has one (cleaned like in the notebook)
        truth = None
        if "Label" in data.columns:
            truth = data["Label"].astype(str).str.strip().str.replace(r"[^\x00-\x7f]+", "-", regex=True).values

        by_type = pd.Series(predicted).value_counts().rename_axis("Type").reset_index(name="Flows")
        summary = pd.DataFrame({
            "Metric": ["Total flows", "Flagged as attack", "Allowed", "Attack %"],
            "Value": [f"{len(X):,}", f"{is_attack.sum():,}", f"{(~is_attack).sum():,}", f"{is_attack.mean():.2%}"],
        })

        st.subheader("Summary")
        st.table(summary)
        st.subheader("Flows by type")
        st.table(by_type)
        if truth is not None:
            st.subheader("Results on this file (ground truth supplied)")
            st.table(overall_table(truth, predicted))

        st.download_button("Download full report (Excel)", make_report(file.name, predicted, truth),
                           "ransomguard_report.xlsx", mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
                           on_click="ignore")

Overwriting app.py


## 11. Run the app
Open the `trycloudflare.com` link it prints.

In [ ]:
!pip install -q streamlit pyarrow openpyxl
!pkill -f streamlit
!streamlit run app.py --server.port 8501 --server.maxUploadSize 1024 &>/dev/null &
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501

2026-11-23T12:09:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-23T12:09:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-23T12:09:34Z INF +--------------------------------------------------------------------------------------------+
2026-11-23T12:09:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-23T12:09:34Z INF |  https://examine-donate-autumn-thee.trycloudflare.com 